# NLP Pipeline - Practice

## Installer les prérequis

In [1]:
!pip install requests beautifulsoup4 selenium webdriver-manager PyPDF2 spacy nltk
!python -m spacy download fr_core_news_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 73.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Exemple 1 :

In [2]:
import requests
from bs4 import BeautifulSoup
import spacy
import PyPDF2
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Chargement du modèle linguistique français de spaCy
# Ce modèle contient les dictionnaires pour le POS tagging, la lemmatisation et le NER
nlp = spacy.load("fr_core_news_sm")

# -----------------------------------------------------------------------------
# ÉTAPE 1 : COLLECTE DES DONNÉES
# Rassembler le texte brut à partir de sources variées (Web, PDF, etc.)
# -----------------------------------------------------------------------------
def collect_data_web(url):
    """Récupère le texte brut depuis une page HTML."""
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    return " ".join([p.text for p in soup.find_all('p')])

# -----------------------------------------------------------------------------
# ÉTAPE 2 : PRÉTRAITEMENT DU TEXTE
# Prépare le texte : Nettoyage, Tokenisation, Normalisation et Réduction
# -----------------------------------------------------------------------------
def preprocess_text(raw_text):
    # A. NETTOYAGE & NORMALISATION : Passage en minuscule et retrait des espaces inutiles
    text = raw_text.lower().strip()

    # B. TOKENISATION & LEMMATISATION : spaCy découpe en tokens et trouve la forme canonique
    doc = nlp(text)

    # C. FILTRAGE : On retire les 'Stopwords' (mots vides) et la ponctuation
    # On ne garde que le 'lemme' (racine linguistique) pour réduire la complexité
    tokens_valides = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]

    return doc, " ".join(tokens_valides)

# -----------------------------------------------------------------------------
# ÉTAPE 3 & 4 : ANALYSE SYNTAXIQUE & SÉMANTIQUE
# Comprendre la structure (POS: Part of Speech) et identifier les entités (NER)
#          --> Part-of-Speech (POS) tagging is the process of assigning grammatical categories
#             (like noun, verb, adjective, etc.) to each word in a sentence based on its definition and context.
# -----------------------------------------------------------------------------
def analyze_linguistics(doc):
    print(f"{'Mot':<12} | {'Nature (POS)':<10} | {'Dépendance'}")
    print("-" * 40)
    for token in list(doc)[:5]: # Exemple sur les 5 premiers mots
        # ÉTAPE 3 : POS Tagging (Nom, Verbe, Adjectif...)
        print(f"{token.text:<12} | {token.pos_:<10} | {token.dep_}")

    # ÉTAPE 4 : Reconnaissance d'Entités Nommées (NER)
    print("\n--- Entités Nommées détectées ---")
    for ent in doc.ents:
        print(f"Entité : {ent.text} ({ent.label_})")

# -----------------------------------------------------------------------------
# ÉTAPE 5 : REPRÉSENTATION DES DONNÉES (VECTORISATION)
# Convertir le texte en nombres pour que la machine puisse "calculer"
# -----------------------------------------------------------------------------
def vectorize_data(corpus):
    # Utilisation de TF-IDF pour donner du poids aux mots significatifs
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(corpus)
    return X, vectorizer

# -----------------------------------------------------------------------------
# ÉTAPE 6 & 7 : MODÉLISATION ET ÉVALUATION
# Appliquer un algorithme et tester ses performances
# -----------------------------------------------------------------------------
'''
def train_and_evaluate(X, y):
    # Exemple de classification (Spam vs Non-Spam)
    model = MultinomialNB()
    model.fit(X, y)

    # Évaluation (Précision, Rappel, F1-score...)
    score = model.score(X, y)
    print(f"\nPrécision du modèle : {score * 100:.2f}%")
    return model
'''
# =============================================================================
# EXÉCUTION DU PIPELINE
# =============================================================================

# Texte d'exemple (Collecte)
raw_content = "Apple a été fondée par Steve Jobs en Californie. Il a créé l'iPhone."

# Prétraitement (Étape 2)
document_spacy, text_clean = preprocess_text(raw_content)

# Analyse (Étape 3 & 4)
analyze_linguistics(document_spacy)

# Représentation (Étape 5)
# (Ici sur un mini-corpus pour l'exemple)
X_vectors, feat_vec = vectorize_data([text_clean])


print("\nPipeline terminé avec succès.")

Mot          | Nature (POS) | Dépendance
----------------------------------------
apple        | NOUN       | nsubj:pass
a            | AUX        | aux:tense
été          | AUX        | aux:pass
fondée       | VERB       | ROOT
par          | ADP        | case

--- Entités Nommées détectées ---
Entité : apple (ORG)
Entité : steve jobs (PER)
Entité : californie (LOC)

Pipeline terminé avec succès.



1. Tokenisation (Le découpage)
Le texte est segmenté en unités atomiques. Le modèle fr_core_news_sm est assez intelligent pour comprendre que "d'ia" doit être séparé en ["d'", "ia"].

2. Stopwords (Le nettoyage)
Le code utilise la propriété .is_stop de spaCy. Les mots comme "les", "de", "la" ou "à" sont évacués car ils sont statistiquement trop fréquents pour aider à classer le texte.

3. Lemmatisation (L'unification)
C'est ici que l'analyse linguistique prend tout son sens. Le mot "étudient" est ramené à "étudier" et "ingénieurs" devient "ingénieur".

Pourquoi ? Si vous avez un autre texte qui parle d'un "ingénieur", le modèle saura qu'il s'agit du même sujet, même si le pluriel diffère.

4. TF-IDF (La signature numérique)
Le tableau final montre le poids de chaque mot.

TF (Term Frequency) : Plus le mot "IA" est présent dans votre texte, plus son score monte.

IDF (Inverse Document Frequency) : Si le mot "FST" est présent dans tous les documents de votre présentation, son score baissera car il n'est plus discriminant (il ne permet plus de différencier un document d'un autre).

Résultat final : Votre phrase est passée d'une structure complexe et humaine à un vecteur de probabilités mathématiques prêt à être injecté dans un classifieur (comme une Régression Logistique ou Naive Bayes).

# Exemple 2:

In [3]:
import spacy
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Chargement du modèle français
nlp = spacy.load("fr_core_news_sm")

def trace_nlp_pipeline(text):
    print(f"--- TEXTE BRUT ---\n'{text}'\n")

    # 1. NORMALISATION & TOKENISATION
    doc = nlp(text.lower().strip())
    tokens_bruts = [token.text for token in doc]
    print(f"1. Tokenisation (Minuscules) :\n{tokens_bruts}\n")

    # 2. FILTRAGE (Stopwords & Ponctuation)
    tokens_filtres = [token.text for token in doc if not token.is_stop and not token.is_punct]
    stopwords_elimines = [token.text for token in doc if token.is_stop]
    print(f"2. Après retrait des Stopwords :\n{tokens_filtres}")
    print(f"(Mots supprimés : {list(set(stopwords_elimines))})\n")

    # 3. LEMMATISATION
    # On transforme chaque mot filtré en sa forme dictionnaire (lemme)
    lemmes = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    print(f"3. Après Lemmatisation (Racines) :\n{lemmes}\n")

    return " ".join(lemmes)

# --- EXEMPLE DÉTAILLÉ ---
phrase_test = "Les ingénieurs étudient les algorithmes d'IA à la FST de Tanger !"
texte_propre = trace_nlp_pipeline(phrase_test)

# --- 4. ZOOM SUR TF-IDF (Vectorisation) ---
# Imaginons un mini-corpus pour calculer les poids
corpus = [texte_propre, "étudiant fst tanger", "algorithme ia complexe"]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# Affichage des scores pour le premier document
print("4. Représentation Numérique (TF-IDF) pour le document 1 :")
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
print(df_tfidf.iloc[0].sort_values(ascending=False))

--- TEXTE BRUT ---
'Les ingénieurs étudient les algorithmes d'IA à la FST de Tanger !'

1. Tokenisation (Minuscules) :
['les', 'ingénieurs', 'étudient', 'les', 'algorithmes', "d'", 'ia', 'à', 'la', 'fst', 'de', 'tanger', '!']

2. Après retrait des Stopwords :
['ingénieurs', 'étudient', 'algorithmes', 'ia', 'fst', 'tanger']
(Mots supprimés : ["d'", 'de', 'à', 'la', 'les'])

3. Après Lemmatisation (Racines) :
['ingénieur', 'étudier', 'algorithme', 'ia', 'fst', 'tanger']

4. Représentation Numérique (TF-IDF) pour le document 1 :
étudier       0.481482
ingénieur     0.481482
algorithme    0.366180
fst           0.366180
tanger        0.366180
ia            0.366180
complexe      0.000000
étudiant      0.000000
Name: 0, dtype: float64


## Interprétation du résultat

Ce résultat montre la transformation complète d'une phrase humaine en une **signature mathématique** que l'IA peut traiter. Voici l'explication détaillée de chaque étape de votre pipeline :



### 1. Tokenisation (Le découpage)
Le texte brut est segmenté en unités minimales appelées **tokens**.
* **Observation :** Le modèle a intelligemment séparé `"d'ia"` en `"d'"` et `"ia"`.
* **Objectif :** Isoler chaque élément syntaxique pour pouvoir les traiter individuellement. La mise en minuscules évite que l'ordinateur ne traite "Les" et "les" comme deux mots différents.


### 2. Retrait des Stopwords (Le filtrage)
On élimine les "mots vides" (déterminants, prépositions).
* **Observation :** Les mots `['de', 'les', 'la', 'à', "d'"]` ont disparu.
* **Objectif :** Réduire le "bruit". Ces mots sont statistiquement trop fréquents dans la langue française pour aider à distinguer le sujet de la phrase. On ne garde que les **mots porteurs de sens** (les mots pleins).


### 3. Lemmatisation (L'unification)
C'est l'étape de normalisation linguistique. On ramène chaque mot à sa forme canonique (celle du dictionnaire).
* **Transformations :** * `ingénieurs` (pluriel) $\rightarrow$ **ingénieur** (singulier).
    * `étudient` (verbe conjugué) $\rightarrow$ **étudier** (infinitif).
* **Objectif :** Regrouper les variantes d'un même mot. Si un autre texte utilisait "ingénieur" au singulier, le modèle comprendrait qu'il s'agit du même concept.


### 4. TF-IDF (La pondération mathématique)
C'est ici que l'on transforme les mots en nombres. Le score **TF-IDF** indique l'importance d'un mot dans votre document par rapport à un ensemble de documents (le corpus).

#### Pourquoi certains scores sont-ils plus hauts (0.48 vs 0.36) ?
Le score dépend de deux facteurs :
1.  **TF (Term Frequency) :** Combien de fois le mot apparaît dans votre phrase.
2.  **IDF (Inverse Document Frequency) :** À quel point le mot est **rare** dans les autres documents de votre exemple.

* **Poids fort (0.48) pour "étudier" et "ingénieur" :** Cela signifie que ces mots sont très spécifiques à votre document 1 et n'apparaissent probablement pas (ou très peu) dans les autres documents de comparaison que vous avez fournis au `TfidfVectorizer`. Ils sont donc jugés **très représentatifs** du contenu.
* **Poids moyen (0.36) pour "ia", "fst", "tanger" :** Ces mots sont importants, mais s'ils apparaissent aussi dans les autres documents du corpus (comme "étudiant fst tanger"), leur score baisse car ils deviennent moins "uniques" pour différencier le document 1 des autres.
* **Score 0.00 pour "complexe" et "étudiant" :** Ces mots n'existent tout simplement pas dans votre phrase de test, même s'ils font partie du vocabulaire global du corpus.



### En bref
Votre phrase n'est plus du texte pour la machine, c'est un **vecteur dans un espace multidimensionnel**. Plus le score est élevé, plus le mot est considéré comme un "mot-clé" crucial pour caractériser votre document.

# Exemple 3

In [4]:
import spacy
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Chargement des outils
nlp = spacy.load("fr_core_news_sm")
stemmer = SnowballStemmer("french")

text_brut = "Les algorithmes d'IA analysent les données massives pour les chercheurs à Tanger."

# --- STEP 1 : Tokenisation & Stopwords ---
doc = nlp(text_brut.lower())
tokens_clean = [t for t in doc if not t.is_stop and not t.is_punct]
print(f"1. Après Stopwords : {[t.text for t in tokens_clean]}")

# --- STEP 2 : Stemming ---
stems = [stemmer.stem(t.text) for t in tokens_clean]
print(f"2. Stemming (Racines) : {stems}")

# --- STEP 3 : Lemmatisation ---
lemmas = [t.lemma_ for t in tokens_clean]
print(f"3. Lemmatisation (Dictionnaire) : {lemmas}")

# --- STEP 4 : TF-IDF ---
# On crée un mini-corpus pour que le calcul TF-IDF ait du sens
corpus = [" ".join(lemmas), "ia tanger fst", "algorithme donnée analyse"]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# Affichage des poids pour notre phrase (Document 0)
print("\n4. Poids TF-IDF (Importance des mots) :")
df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
print(df.iloc[0].sort_values(ascending=False))

1. Après Stopwords : ['algorithmes', 'ia', 'analysent', 'données', 'massives', 'chercheurs', 'tanger']
2. Stemming (Racines) : ['algorithm', 'ia', 'analysent', 'don', 'massiv', 'chercheur', 'tang']
3. Lemmatisation (Dictionnaire) : ['algorithme', 'ia', 'analyser', 'donnée', 'massif', 'chercheur', 'tanger']

4. Poids TF-IDF (Importance des mots) :
analyser      0.433816
massif        0.433816
chercheur     0.433816
algorithme    0.329928
donnée        0.329928
tanger        0.329928
ia            0.329928
analyse       0.000000
fst           0.000000
Name: 0, dtype: float64


# Devoir

- Extraire les données depuis un pdf
- Utiliser des textes arabes au lieu de francais

In [5]:
import PyPDF2

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text()
    return text

# Exemple de texte extrait (Brut)
raw_pdf_text = "Les chercheurs étudient l'intelligence artificielle. Ils analysent les données massives."

# Compléter le code, modifier si besoin

---
#  Partie Arabe

## Défis spécifiques de l'arabe en NLP

La langue arabe présente des caractéristiques uniques qui compliquent le traitement NLP :

| Défi | Explication | Exemple |
|------|------------|--------|
| **Écriture RTL** | De droite à gauche | ← الكتابة بالعربية |
| **Morphologie riche** | Un mot = plusieurs formes | كتب، كاتب، مكتوب، كتابة |
| **Diacritiques (Tashkeel)** | Voyelles optionnelles | كَتَبَ vs كتب |
| **Pas d'espaces entre certains mots** | Les prépositions collent au mot | في+المدرسة = في المدرسة |
| **Dialectes** | Darija ≠ MSA (arabe standard) | درنا vs فعلنا |

## Outils utilisés
- **`pyarabic`** : Nettoyage, normalisation, suppression des diacritiques
- **`nltk` (ISRIStemmer)** : Stemming de l'arabe
- **`scikit-learn` (TfidfVectorizer)** : Vectorisation
- **`PyPDF2`** : Extraction du texte depuis un PDF

In [6]:
# Installation des bibliothèques nécessaires pour l'arabe
!pip install pyarabic nltk scikit-learn PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 2.7 MB/s eta 0:00:00


## Étape 1 — Extraction du texte depuis un PDF arabe

In [7]:
import PyPDF2

def extract_arabic_from_pdf(pdf_path):
    """
    Extrait le texte arabe depuis un fichier PDF.
    Retourne le texte brut de toutes les pages.
    """
    extracted_text = ""
    try:
        with open(pdf_path, "rb") as pdf_file:
            reader = PyPDF2.PdfReader(pdf_file)
            print(f"Nombre de pages détectées : {len(reader.pages)}")
            for i, page in enumerate(reader.pages):
                page_text = page.extract_text()
                if page_text:
                    extracted_text += page_text + "\n"
                    print(f"  Page {i+1} : {len(page_text)} caractères extraits")
    except FileNotFoundError:
        print(f"Fichier non trouvé : {pdf_path}")
        print("→ Utilisation du texte de démonstration intégré.")
    return extracted_text

# -----------------------------------------------------------------
# Essayez avec votre propre PDF arabe :
# pdf_text = extract_arabic_from_pdf("mon_document_arabe.pdf")
# -----------------------------------------------------------------

# Texte de démonstration (simulant un PDF extrait)
pdf_text_demo = """
تُعدّ الذكاء الاصطناعي من أبرز مجالات العلوم الحديثة.
يعمل الباحثون في جامعة طنجة على تطوير نماذج لمعالجة اللغة العربية الطبيعية.
درس الطلاب خوارزميات تعلم الآلة في الجامعة وحققوا نتائج متميزة.
يُحلّل النظام البيانات الضخمة باستخدام نماذج متقدمة ومتطورة.
أسّس ستيف جوبز شركة أبل في ولاية كاليفورنيا الأمريكية.
تتكوّن معالجة اللغات الطبيعية من عدة مراحل أساسية تشمل التجزيء والتطبيع.
"""

print("=" * 50)
print("TEXTE EXTRAIT DU PDF (démonstration) :")
print("=" * 50)
print(pdf_text_demo)
print(f"Longueur totale : {len(pdf_text_demo)} caractères")

TEXTE EXTRAIT DU PDF (démonstration) :

تُعدّ الذكاء الاصطناعي من أبرز مجالات العلوم الحديثة.
يعمل الباحثون في جامعة طنجة على تطوير نماذج لمعالجة اللغة العربية الطبيعية.
درس الطلاب خوارزميات تعلم الآلة في الجامعة وحققوا نتائج متميزة.
يُحلّل النظام البيانات الضخمة باستخدام نماذج متقدمة ومتطورة.
أسّس ستيف جوبز شركة أبل في ولاية كاليفورنيا الأمريكية.
تتكوّن معالجة اللغات الطبيعية من عدة مراحل أساسية تشمل التجزيء والتطبيع.

Longueur totale : 384 caractères


## Étape 2 — Nettoyage et normalisation du texte arabe

> **Pourquoi normaliser l'arabe ?**  
> Le mot `أنا` (moi) peut s'écrire `انا` ou `أنا` selon le document.  
> La normalisation uniformise toutes les variantes orthographiques pour que le modèle les traite comme le **même mot**.

In [8]:
import re
import pyarabic.araby as araby

def normalize_arabic(text):
    """
    Normalisation complète du texte arabe :
    1. Suppression des diacritiques (تشكيل)
    2. Normalisation des Alef (أ إ آ → ا)
    3. Suppression de la Tatweel (ـ)
    4. Normalisation du Teh Marbuta (ة → ه)
    5. Suppression des caractères non arabes et espaces multiples
    """
    # 1. Suppression des diacritiques (les petites voyelles au-dessus/dessous)
    text = araby.strip_tashkeel(text)

    # 2. Normalisation des différentes formes de la lettre Alef
    #    أ إ آ ٱ → ا (forme simple)
    text = araby.normalize_alef(text)

    # 3. Suppression de la Tatweel (la ligne d'allongement ـ)
    text = araby.strip_tatweel(text)

    # 4. Normalisation du Teh Marbuta (ة → ه)
    text = araby.normalize_teh(text)

    # 5. Suppression de tout ce qui n'est pas arabe ou espace
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)

    # 6. Suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# --- DÉMONSTRATION ---
exemples_avant_apres = [
    ("تُعَلَّمُ الطُّلَّابُ",  "Avec diacritiques → sans diacritiques"),
    ("أُسْتَاذٌ أَجْنَبِيٌّ",  "Alef normalisée"),
    ("مَدْرَسَـــة",           "Tatweel supprimée"),
    ("جَامِعَة طَنْجَة",       "Teh Marbuta normalisée"),
]

print("DÉMONSTRATION DE LA NORMALISATION")
print("-" * 55)
for texte_ar, description in exemples_avant_apres:
    texte_norm = normalize_arabic(texte_ar)
    print(f"  [{description}]")
    print(f"  Avant  : {texte_ar}")
    print(f"  Après  : {texte_norm}")
    print()

# Normalisation du texte PDF
texte_normalise = normalize_arabic(pdf_text_demo)
print("=" * 55)
print("TEXTE PDF NORMALISÉ :")
print(texte_normalise)

DÉMONSTRATION DE LA NORMALISATION
-------------------------------------------------------
  [Avec diacritiques → sans diacritiques]
  Avant  : تُعَلَّمُ الطُّلَّابُ
  Après  : تعلم الطلاب

  [Alef normalisée]
  Avant  : أُسْتَاذٌ أَجْنَبِيٌّ
  Après  : استاذ اجنبي

  [Tatweel supprimée]
  Avant  : مَدْرَسَـــة
  Après  : مدرسه

  [Teh Marbuta normalisée]
  Avant  : جَامِعَة طَنْجَة
  Après  : جامعه طنجه

TEXTE PDF NORMALISÉ :
تعد الذكاء الاصطناعي من ابرز مجالات العلوم الحديثه يعمل الباحثون في جامعه طنجه علا تطوير نماذج لمعالجه اللغه العربيه الطبيعيه درس الطلاب خوارزميات تعلم الاله في الجامعه وحققوا نتائج متميزه يحلل النظام البيانات الضخمه باستخدام نماذج متقدمه ومتطوره اسس ستيف جوبز شركه ابل في ولايه كاليفورنيا الامريكيه تتكون معالجه اللغات الطبيعيه من عده مراحل اساسيه تشمل التجزيء والتطبيع


## Étape 3 — Tokenisation et Stopwords arabes

> En arabe, les prépositions et conjonctions (`في`, `من`, `إلى`, `على`, `هو`...) sont aussi des stopwords à éliminer, tout comme `les`, `de`, `à` en français.

In [9]:
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

def tokenize_arabic(text):
    """
    Tokenise le texte arabe (découpage par espaces et ponctuation).
    L'arabe ne nécessite pas de tokeniseur complexe pour les cas simples.
    """
    # Découpage sur les espaces (le texte a déjà été nettoyé)
    tokens = text.split()
    # Filtrage des tokens vides et trop courts (< 2 caractères)
    tokens = [t for t in tokens if len(t) >= 2]
    return tokens

def remove_arabic_stopwords(tokens):
    """
    Supprime les mots vides (stopwords) de la liste de tokens.
    Utilise la liste NLTK + des stopwords supplémentaires.
    """
    # Liste NLTK des stopwords arabes
    nltk_stopwords = set(stopwords.words('arabic'))

    # Stopwords supplémentaires courants
    extra_stopwords = {
        'من', 'في', 'على', 'إلى', 'هو', 'هي', 'هم', 'هن',
        'ان', 'اي', 'اذا', 'لكن', 'لا', 'ما', 'قد', 'كان',
        'مع', 'عن', 'بين', 'حيث', 'كما', 'بعد', 'قبل', 'كل',
        'التي', 'الذي', 'الذين', 'التي', 'ذلك', 'هذا', 'هذه'
    }

    all_stopwords = nltk_stopwords | extra_stopwords

    tokens_filtres = [t for t in tokens if t not in all_stopwords]
    stopwords_trouves = [t for t in tokens if t in all_stopwords]

    return tokens_filtres, stopwords_trouves

# --- EXEMPLE PAS À PAS ---
phrase_test_ar = "يعمل الباحثون في جامعه طنجه على تطوير نماذج للذكاء الاصطناعي"

print("TEXTE BRUT :")
print(f"  {phrase_test_ar}\n")

# Normalisation
phrase_norm = normalize_arabic(phrase_test_ar)
print("APRÈS NORMALISATION :")
print(f"  {phrase_norm}\n")

# Tokenisation
tokens = tokenize_arabic(phrase_norm)
print("TOKENS :")
print(f"  {tokens}\n")

# Stopwords
tokens_propres, mots_supprimes = remove_arabic_stopwords(tokens)
print("APRÈS SUPPRESSION DES STOPWORDS :")
print(f"  Gardés    : {tokens_propres}")
print(f"  Supprimés : {mots_supprimes}")

TEXTE BRUT :
  يعمل الباحثون في جامعه طنجه على تطوير نماذج للذكاء الاصطناعي

APRÈS NORMALISATION :
  يعمل الباحثون في جامعه طنجه علا تطوير نماذج للذكاء الاصطناعي

TOKENS :
  ['يعمل', 'الباحثون', 'في', 'جامعه', 'طنجه', 'علا', 'تطوير', 'نماذج', 'للذكاء', 'الاصطناعي']

APRÈS SUPPRESSION DES STOPWORDS :
  Gardés    : ['يعمل', 'الباحثون', 'جامعه', 'طنجه', 'علا', 'تطوير', 'نماذج', 'للذكاء', 'الاصطناعي']
  Supprimés : ['في']


## Étape 4 — Stemming arabe (ISRIStemmer)

> **ISRI Stemmer** = *Information Science Research Institute's Arabic Stemmer*  
> C'est le stemmer arabe intégré à NLTK. Il supprime les préfixes et suffixes pour trouver la **racine** (الجذر) du mot.
>
> Exemple : `الباحثون` (les chercheurs) → `بحث` (chercher/recherche)

In [10]:
from nltk.stem.isri import ISRIStemmer

def stem_arabic(tokens):
    """
    Applique le stemming arabe sur une liste de tokens.
    Utilise l'ISRIStemmer de NLTK (racines tri-consonantiques).
    """
    stemmer = ISRIStemmer()
    stems = [stemmer.stem(token) for token in tokens]
    return stems

# --- DÉMONSTRATION DES RACINES ---
mots_demo = [
    "الباحثون",   # Les chercheurs
    "يدرسون",    # Ils étudient
    "خوارزميات", # Algorithmes
    "البيانات",  # Les données
    "متقدمة",   # Avancées
    "جامعه",    # Université
    "طنجه",     # Tanger
    "الاصطناعي",# Artificielle
]

stemmer = ISRIStemmer()

print(f"{'Mot original':<20} | {'Racine (Stem)':<15} | Traduction")
print("-" * 60)
traductions = [
    "Les chercheurs", "Ils étudient", "Algorithmes", "Les données",
    "Avancées", "Université", "Tanger", "Artificielle"
]
for mot, trad in zip(mots_demo, traductions):
    racine = stemmer.stem(normalize_arabic(mot))
    print(f"  {mot:<18} | {racine:<15} | {trad}")

# --- APPLICATION SUR LE TEXTE COMPLET ---
print("\n" + "=" * 55)
print("PIPELINE COMPLET SUR LA PHRASE TEST :")
print("=" * 55)

phrase_complete = "يعمل الباحثون في جامعة طنجة على تطوير نماذج للذكاء الاصطناعي"
texte_n = normalize_arabic(phrase_complete)
tokens_all = tokenize_arabic(texte_n)
tokens_sans_stop, _ = remove_arabic_stopwords(tokens_all)
stems_final = stem_arabic(tokens_sans_stop)

print(f"Brut        : {phrase_complete}")
print(f"Normalisé   : {texte_n}")
print(f"Tokens      : {tokens_sans_stop}")
print(f"Stems       : {stems_final}")

Mot original         | Racine (Stem)   | Traduction
------------------------------------------------------------
  الباحثون           | بحث             | Les chercheurs
  يدرسون             | درس             | Ils étudient
  خوارزميات          | خوارزم          | Algorithmes
  البيانات           | بين             | Les données
  متقدمة             | تقدم            | Avancées
  جامعه              | جمع             | Université
  طنجه               | طنج             | Tanger
  الاصطناعي          | صطناع           | Artificielle

PIPELINE COMPLET SUR LA PHRASE TEST :
Brut        : يعمل الباحثون في جامعة طنجة على تطوير نماذج للذكاء الاصطناعي
Normalisé   : يعمل الباحثون في جامعه طنجه علا تطوير نماذج للذكاء الاصطناعي
Tokens      : ['يعمل', 'الباحثون', 'جامعه', 'طنجه', 'علا', 'تطوير', 'نماذج', 'للذكاء', 'الاصطناعي']
Stems       : ['عمل', 'بحث', 'جمع', 'طنج', 'علا', 'طور', 'اذج', 'ذكء', 'صطناع']


## Étape 5 — Pipeline Complet & Vectorisation TF-IDF

On assemble toutes les étapes et on vectorise un corpus arabe complet.

In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem.isri import ISRIStemmer

stemmer_ar = ISRIStemmer()

def arabic_nlp_pipeline(text, verbose=True):
    """
    Pipeline NLP complet pour le texte arabe :
    Normalisation → Tokenisation → Stopwords → Stemming
    Retourne la chaîne de stems prête pour TF-IDF.
    """
    if verbose:
        print(f"[1] BRUT      : {text[:80]}..." if len(text) > 80 else f"[1] BRUT      : {text}")

    # Étape 1 : Normalisation
    normalized = normalize_arabic(text)
    if verbose:
        print(f"[2] NORMALISÉ : {normalized[:80]}..." if len(normalized) > 80 else f"[2] NORMALISÉ : {normalized}")

    # Étape 2 : Tokenisation
    tokens = tokenize_arabic(normalized)
    if verbose:
        print(f"[3] TOKENS    : {tokens}")

    # Étape 3 : Suppression des stopwords
    tokens_clean, removed = remove_arabic_stopwords(tokens)
    if verbose:
        print(f"[4] -STOPS    : {tokens_clean}  (supprimés: {removed})")

    # Étape 4 : Stemming
    stems = [stemmer_ar.stem(t) for t in tokens_clean]
    if verbose:
        print(f"[5] STEMS     : {stems}")
        print()

    return " ".join(stems)

# =========================================================
# CORPUS ARABE (simulant des documents extraits d'un PDF)
# =========================================================
corpus_arabe = [
    "يعمل الباحثون في جامعة طنجة على تطوير نماذج لمعالجة اللغة العربية الطبيعية",
    "درس الطلاب خوارزميات تعلم الآلة وحققوا نتائج متميزة في مجال الذكاء الاصطناعي",
    "يحلل النظام البيانات الضخمة باستخدام خوارزميات متقدمة ونماذج الشبكات العصبية",
    "تتكون معالجة اللغات الطبيعية من مراحل أساسية تشمل التجزيء والتطبيع والتصنيف",
    "تطوير تطبيقات الذكاء الاصطناعي يتطلب بيانات ضخمة ونماذج تعلم آلة متطورة",
]

print("=" * 60)
print("PIPELINE NLP ARABE — TRAITEMENT DU CORPUS")
print("=" * 60)

corpus_stems = []
for i, doc in enumerate(corpus_arabe):
    print(f"--- Document {i+1} ---")
    processed = arabic_nlp_pipeline(doc, verbose=True)
    corpus_stems.append(processed)

# =========================================================
# VECTORISATION TF-IDF
# =========================================================
print("=" * 60)
print("VECTORISATION TF-IDF DU CORPUS ARABE")
print("=" * 60)

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus_stems)

df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus_stems))]
)

print("\nTop 8 mots les plus importants par document :")
for idx in df_tfidf.index:
    top_words = df_tfidf.loc[idx].sort_values(ascending=False).head(8)
    top_str = ", ".join([f"{w}({v:.2f})" for w, v in top_words.items() if v > 0])
    print(f"  {idx}: {top_str}")

print("\nMatrice TF-IDF complète :")
print(df_tfidf.round(3).to_string())

PIPELINE NLP ARABE — TRAITEMENT DU CORPUS
--- Document 1 ---
[1] BRUT      : يعمل الباحثون في جامعة طنجة على تطوير نماذج لمعالجة اللغة العربية الطبيعية
[2] NORMALISÉ : يعمل الباحثون في جامعه طنجه علا تطوير نماذج لمعالجه اللغه العربيه الطبيعيه
[3] TOKENS    : ['يعمل', 'الباحثون', 'في', 'جامعه', 'طنجه', 'علا', 'تطوير', 'نماذج', 'لمعالجه', 'اللغه', 'العربيه', 'الطبيعيه']
[4] -STOPS    : ['يعمل', 'الباحثون', 'جامعه', 'طنجه', 'علا', 'تطوير', 'نماذج', 'لمعالجه', 'اللغه', 'العربيه', 'الطبيعيه']  (supprimés: ['في'])
[5] STEMS     : ['عمل', 'بحث', 'جمع', 'طنج', 'علا', 'طور', 'اذج', 'علج', 'لغه', 'عرب', 'طبع']

--- Document 2 ---
[1] BRUT      : درس الطلاب خوارزميات تعلم الآلة وحققوا نتائج متميزة في مجال الذكاء الاصطناعي
[2] NORMALISÉ : درس الطلاب خوارزميات تعلم الاله وحققوا نتائج متميزه في مجال الذكاء الاصطناعي
[3] TOKENS    : ['درس', 'الطلاب', 'خوارزميات', 'تعلم', 'الاله', 'وحققوا', 'نتائج', 'متميزه', 'في', 'مجال', 'الذكاء', 'الاصطناعي']
[4] -STOPS    : ['درس', 'الطلاب', 'خوارزميات', 'تعلم', '

## Étape 6 — Pipeline complet : PDF arabe → TF-IDF

Cette cellule combine **extraction PDF** + **pipeline NLP arabe** en une seule fonction.

In [14]:
import PyPDF2
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

def full_arabic_pipeline_from_pdf(pdf_path=None, demo_text=None):
    """
    Pipeline complet : PDF arabe → Texte → NLP → TF-IDF

    Args:
        pdf_path  : chemin vers le fichier PDF (optionnel)
        demo_text : texte de démonstration si pas de PDF
    """

    # ---- ÉTAPE 1 : Extraction ----
    print("📄 ÉTAPE 1 : Extraction du texte")
    if pdf_path:
        raw_text = extract_arabic_from_pdf(pdf_path)
    else:
        raw_text = demo_text or pdf_text_demo
        print("  (Mode démonstration - pas de PDF fourni)")
    print(f"  → {len(raw_text)} caractères extraits\n")

    # ---- ÉTAPE 2 : Découpage en phrases ----
    print("✂️  ÉTAPE 2 : Découpage en phrases")
    # Découpage sur les points et les nouvelles lignes
    import re
    phrases = re.split(r'[.\n]+', raw_text)
    phrases = [p.strip() for p in phrases if len(p.strip()) > 10]
    print(f"  → {len(phrases)} phrases détectées\n")

    # ---- ÉTAPE 3 : Pipeline NLP ----
    print("🔧 ÉTAPE 3 : Traitement NLP (normalisation + stems)")
    corpus_traite = []
    for i, phrase in enumerate(phrases[:6]):  # Limiter à 6 pour l'affichage
        processed = arabic_nlp_pipeline(phrase, verbose=False)
        if processed:
            corpus_traite.append(processed)
            print(f"  Phrase {i+1}: {phrase[:50]}...")
            print(f"           → {processed}")
    print()

    # ---- ÉTAPE 4 : Vectorisation TF-IDF ----
    print("📊 ÉTAPE 4 : Vectorisation TF-IDF")
    if len(corpus_traite) < 2:
        print("  ⚠️  Pas assez de documents pour TF-IDF (minimum 2)")
        return

    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus_traite)

    print(f"  Vocabulaire : {len(vectorizer.get_feature_names_out())} mots uniques")
    print(f"  Matrice     : {tfidf_matrix.shape[0]} documents × {tfidf_matrix.shape[1]} features")

    df = pd.DataFrame(
        tfidf_matrix.toarray(),
        columns=vectorizer.get_feature_names_out(),
        index=[f"Phrase {i+1}" for i in range(len(corpus_traite))]
    )

    print("\n  Top 5 termes les plus discriminants :")
    for idx in df.index:
        top5 = df.loc[idx].sort_values(ascending=False).head(5)
        top_str = ", ".join([f"{w}({v:.2f})" for w, v in top5.items() if v > 0])
        print(f"    {idx}: {top_str}")

    print("\n✅ Pipeline terminé avec succès !")
    return df

# =========================================================
# LANCEMENT DU PIPELINE COMPLET
# =========================================================
print("=" * 60)
print("  PIPELINE NLP ARABE COMPLET : PDF → TF-IDF")
print("=" * 60)
print()


result_df = full_arabic_pipeline_from_pdf(pdf_path="/content/paragraphe_nlp_arabe.pdf")

print()

  PIPELINE NLP ARABE COMPLET : PDF → TF-IDF

📄 ÉTAPE 1 : Extraction du texte
Nombre de pages détectées : 1
  Page 1 : 913 caractères extraits
  → 914 caractères extraits

✂️  ÉTAPE 2 : Découpage en phrases
  → 20 phrases détectées

🔧 ÉTAPE 3 : Traitement NLP (normalisation + stems)

📊 ÉTAPE 4 : Vectorisation TF-IDF
  ⚠️  Pas assez de documents pour TF-IDF (minimum 2)

